In [3]:
import pyspark
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [7]:
# Question 1
print(f"Spark version: {spark.version}")

Spark version: 3.5.8


In [2]:
!mkdir -p hwdata/raw

In [3]:
!mkdir -p hwdata/pq

In [4]:
!wget -P hwdata/raw https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 08:57:17--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.67.246.186, 18.67.246.167, 18.67.246.176, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.67.246.186|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘hwdata/raw/yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  34.4MB/s    in 2.0s    

2026-03-09 08:57:19 (34.4 MB/s) - ‘hwdata/raw/yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [12]:
df_trips = spark.read.parquet('hwdata/raw/yellow_tripdata_2025-11.parquet')

In [13]:
df_trips.repartition(4).write.parquet('hwdata/pq', mode='overwrite')

In [14]:
# Question 2
!ls -lh hwdata/pq/*.parquet

-rw-r--r-- 1 spark spark 25M Mar  9 19:32 hwdata/pq/part-00000-b795e4c8-fc6e-44ac-b933-fa3ff408afe8-c000.snappy.parquet
-rw-r--r-- 1 spark spark 25M Mar  9 19:32 hwdata/pq/part-00001-b795e4c8-fc6e-44ac-b933-fa3ff408afe8-c000.snappy.parquet
-rw-r--r-- 1 spark spark 25M Mar  9 19:32 hwdata/pq/part-00002-b795e4c8-fc6e-44ac-b933-fa3ff408afe8-c000.snappy.parquet
-rw-r--r-- 1 spark spark 25M Mar  9 19:32 hwdata/pq/part-00003-b795e4c8-fc6e-44ac-b933-fa3ff408afe8-c000.snappy.parquet


In [15]:
from pyspark.sql import functions as F

In [21]:
# Question 3
df_trips \
    .withColumn('pickup_date', F.to_date(df_trips.tpep_pickup_datetime)) \
    .filter(df_trips.pickup_date == '2025-11-15') \
    .count()

162604

In [30]:
from pyspark.sql import types
from datetime import datetime

In [31]:
df_trips.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'pickup_date']

In [63]:
# Question 4
df_trips \
    .withColumn("hours", (df_trips.tpep_dropoff_datetime - df_trips.tpep_pickup_datetime).cast("long") / 3600) \
    .withColumn('pickup_date', F.to_date(df_trips.tpep_pickup_datetime)) \
    .groupBy('pickup_date') \
    .max('hours') \
    .orderBy('max(hours)', ascending=False) \
    .limit(1) \
    .show()

+-----------+-----------------+
|pickup_date|       max(hours)|
+-----------+-----------------+
| 2025-11-26|90.64666666666666|
+-----------+-----------------+



In [64]:
!wget -P hwdata/raw https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 20:02:17--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.67.246.167, 18.67.246.186, 18.67.246.176, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.67.246.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘hwdata/raw/taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-09 20:02:17 (10.9 MB/s) - ‘hwdata/raw/taxi_zone_lookup.csv’ saved [12331/12331]



In [65]:
df_zones = spark.read.csv('hwdata/raw/taxi_zone_lookup.csv', header=True)

In [66]:
df_trips.createOrReplaceTempView('trips')
df_zones.createOrReplaceTempView('zones')

In [74]:
# Question 6
df_result = df_zones.alias("z") \
    .join(df_trips.alias("t"), F.col("z.LocationID") == F.col("t.PULocationID"), "left") \
    .groupBy("z.Zone") \
    .agg(F.count("t.PULocationID").alias("trip_count")) \
    .orderBy("trip_count")

df_result.show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Charleston/Tottenville                       |0         |
|Freshkills Park                              |0         |
|Great Kills Park                             |0         |
|Governor's Island/Ellis Island/Liberty Island|1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Arden Heights                                |1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
|Rossville/Woodrow                            |4         |
|Great Kills                                  |4         |
|Green-Wood Cemetery                          |4         |
|Jamaica Bay                                  |5         |
|Westerleigh                                  |12        |
|West Brighton                                |14       